<a href="https://colab.research.google.com/github/BenjaminNaveas/mlp-mlflow-breast-cancer/blob/main/IEI097_Lab2_12_MLP_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Actividad 2.12 — Laboratorio: Implementación de Redes Neuronales (MLP + MLflow)
**Estudiante:** Benjamín Eduardo Naveas Molina | **Número de lista:** [Escribe tu número aquí]
**Asignatura:** IEI097 · Minería de Datos | **Fecha:** 05 de Noviembre de 2026

In [9]:
# FASE 1: Entorno y fijación de semilla
!pip install -q mlflow pyngrok

import os, json, random, time, requests, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata
import kagglehub
import mlflow
from pyngrok import ngrok
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay
import warnings; warnings.filterwarnings('ignore')

# ⚠️ IMPORTANTE: Cambia este 15 por tu número de lista real
SEED = 15

random.seed(SEED)
np.random.seed(SEED)

print(f"Entorno configurado. pandas: {pd.__version__}, mlflow: {mlflow.__version__}")

# Configurar credenciales Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": userdata.get('KAGGLE_USERNAME'), "key": userdata.get('KAGGLE_KEY')}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

Entorno configurado. pandas: 2.2.3, mlflow: 3.18.0


In [10]:
# FASE 1: Carga y preparación de datos
path_bc = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
df = pd.read_csv(f"{path_bc}/data.csv")

# Eliminar nulos y columnas basura
print("Nulos en Unnamed: 32 ->", df['Unnamed: 32'].isna().sum())
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

# Codificar diagnosis (Maligno=1, Benigno=0)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
print("\nBalance de clases:\n", df['diagnosis'].value_counts())

# Evidencia de escala
r_mean = df['radius_mean'].agg(['min', 'max'])
a_mean = df['area_mean'].agg(['min', 'max'])
s_mean = df['smoothness_mean'].agg(['min', 'max'])

print("\n--- Evidencia de Escala ---")
print(f"Radius mean: {r_mean['min']} a {r_mean['max']}")
print(f"Area mean: {a_mean['min']} a {a_mean['max']}")
print(f"Smoothness mean: {s_mean['min']} a {s_mean['max']}")
print(f"El rango de Area Mean (aprox 2371) es casi {(2371/0.11):.0f} veces mayor que el rango de Smoothness Mean (aprox 0.11).")

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Nulos en Unnamed: 32 -> 569

Balance de clases:
 diagnosis
0    357
1    212
Name: count, dtype: int64

--- Evidencia de Escala ---
Radius mean: 6.981 a 28.11
Area mean: 143.5 a 2501.0
Smoothness mean: 0.05263 a 0.1634
El rango de Area Mean (aprox 2371) es casi 21555 veces mayor que el rango de Smoothness Mean (aprox 0.11).


In [11]:
from sklearn.model_selection import train_test_split

# FASE 1: Partición estratificada (Sin fuga de datos)
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# Impresión corregida usando f-strings
print(f"Proporción Malignos Train: {y_train.mean():.4f}")
print(f"Proporción Malignos Test: {y_test.mean():.4f}")

Proporción Malignos Train: 0.3736
Proporción Malignos Test: 0.3684


In [13]:
import os
import subprocess
import time
import requests
from google.colab import userdata  # Asumiendo que usas Google Colab por userdata
import mlflow
from pyngrok import ngrok

# FASE 2: Despliegue de MLflow
env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS="*",
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")

# Ejecutamos el servidor redirigiendo stdout/stderr para que no bloquee
subprocess.Popen(["mlflow", "server",
                  "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"],
                 env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Esperando a que MLflow Server responda...")
for i in range(60):
    try:
        # Se cambia el endpoint a la raíz '/' o '/health' según la versión de MLflow
        if requests.get("http://127.0.0.1:5000/", timeout=2).ok:
            print(f"¡Servidor listo en el intento {i+1}!")
            break
    except requests.RequestException:
        # Captura ConnectionError, ReadTimeout, etc.
        pass
    time.sleep(1)

# Configuración de Ngrok
import os
archivo_ngrok = '/root/.config/ngrok/ngrok.yml'
if os.path.exists(archivo_ngrok):
    os.remove(archivo_ngrok) # Borra la configuración anterior si existe

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
tunel = ngrok.connect(5000)
print("🔗 MLflow UI (Abre este link en otra pestaña):", tunel.public_url)

# Configuración de tracking de MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Naveas")

Esperando a que MLflow Server responda...
¡Servidor listo en el intento 1!
🔗 MLflow UI (Abre este link en otra pestaña): https://hunting-reach-virtuous.ngrok-free.dev


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794278596607, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794278596607, lifecycle_stage='active', name='IEI097-L2.12-Naveas', tags={}, trace_location=None, workspace='default'>

# 📝 Hipótesis (Runs A vs B)
**Hipótesis:** El escalado de variables (Run B) mejorará significativamente el rendimiento y reducirá el tiempo de convergencia frente a no escalar (Run A). Como vimos en la evidencia de escala, el Área tiene valores en miles y la Suavidad (Smoothness) en decimales. Sin escalar, el descenso del gradiente estará dominado por el Área, saturando las funciones de activación e impidiendo que el MLP aprenda correctamente.

In [14]:
# FASE 3: Función reutilizable
def correr_experimento(nombre_run, usar_scaler, hidden_layers, activacion, semilla_modelo, fase):
    with mlflow.start_run(run_name=nombre_run):
        # 1. Pipeline
        pasos = []
        if usar_scaler:
            pasos.append(('scaler', StandardScaler()))

        mlp = MLPClassifier(hidden_layer_sizes=hidden_layers, activation=activacion,
                            max_iter=500, random_state=semilla_modelo)
        pasos.append(('mlp', mlp))
        modelo = Pipeline(pasos)

        # 2. Entrenar y Predecir
        modelo.fit(X_train, y_train)
        y_pred_train = modelo.predict(X_train)
        y_pred_test = modelo.predict(X_test)

        # 3. Log Parámetros
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("activation", activacion)
        mlflow.log_param("scaler", usar_scaler)
        mlflow.log_param("random_state", semilla_modelo)
        mlflow.set_tag("fase", fase)

        # 4. Log Métricas
        acc_train = accuracy_score(y_train, y_pred_train)
        acc_test = accuracy_score(y_test, y_pred_test)
        rec = recall_score(y_test, y_pred_test) # Recall maligno
        f1 = f1_score(y_test, y_pred_test)
        iteraciones = modelo.named_steps['mlp'].n_iter_

        mlflow.log_metric("acc_train", acc_train)
        mlflow.log_metric("acc_test", acc_test)
        mlflow.log_metric("recall_maligno", rec)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("iteraciones", iteraciones)

        # 5. Log Curva de pérdida
        curva_perdida = modelo.named_steps['mlp'].loss_curve_
        for epoca, perdida in enumerate(curva_perdida):
            mlflow.log_metric("train_loss", perdida, step=epoca)

        # 6. Log Artefacto (Matriz)
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, ax=ax, display_labels=['Benigno', 'Maligno'], cmap='Blues')
        plt.title(f"Matriz Confusión - {nombre_run}")
        fig.savefig(f"cm_{nombre_run}.png")
        mlflow.log_figure(fig, f"cm_{nombre_run}.png")
        plt.close(fig)

        print(f"[{nombre_run}] Acc: {acc_test:.4f} | Recall: {rec:.4f} | Iters: {iteraciones}")
        return acc_test, rec

In [15]:
# FASE 3: Ejecución de los 4 runs principales
print("Iniciando experimentos principales...")
correr_experimento("Run_A", False, (30,), 'relu', SEED, "experimentos")
correr_experimento("Run_B", True, (30,), 'relu', SEED, "experimentos")
correr_experimento("Run_C", True, (64, 32), 'relu', SEED, "experimentos")
correr_experimento("Run_D", True, (64, 32), 'tanh', SEED, "experimentos")

Iniciando experimentos principales...
[Run_A] Acc: 0.9561 | Recall: 0.8810 | Iters: 201
🏃 View run Run_A at: http://127.0.0.1:5000/#/experiments/1/runs/2c9a2b88ae3a4c838290876bf6e7a328
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_B] Acc: 0.9825 | Recall: 0.9524 | Iters: 428
🏃 View run Run_B at: http://127.0.0.1:5000/#/experiments/1/runs/e1802623f2aa48f1bea3a4562a7c1c37
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_C] Acc: 0.9737 | Recall: 0.9286 | Iters: 184
🏃 View run Run_C at: http://127.0.0.1:5000/#/experiments/1/runs/12335d37dd1043c88b51d3bcee3524f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_D] Acc: 0.9737 | Recall: 0.9524 | Iters: 292
🏃 View run Run_D at: http://127.0.0.1:5000/#/experiments/1/runs/930abe0b7d604b32a0938ab60302ed2b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


(0.9736842105263158, 0.9523809523809523)

In [16]:
# FASE 4: Variabilidad con Semillas
print("\nIniciando prueba de variabilidad con semillas (Config C)...")
accs = []
recs = []

for i in range(5):
    sem = SEED + i
    a, r = correr_experimento(f"Run_Semilla_{sem}", True, (64, 32), 'relu', sem, "semillas")
    accs.append(a)
    recs.append(r)

print(f"\nVariabilidad Exactitud: Rango de {min(accs):.4f} a {max(accs):.4f}")
print(f"Variabilidad Recall: Rango de {min(recs):.4f} a {max(recs):.4f}")

# Replicación exacta
print("\nIniciando réplica exacta (Debe coincidir con Run_C)...")
correr_experimento("Run_Replica_C", True, (64, 32), 'relu', SEED, "replica")


Iniciando prueba de variabilidad con semillas (Config C)...
[Run_Semilla_15] Acc: 0.9737 | Recall: 0.9286 | Iters: 184
🏃 View run Run_Semilla_15 at: http://127.0.0.1:5000/#/experiments/1/runs/f202c3ad1a4f4fbe81485b71590c3f2e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_16] Acc: 0.9825 | Recall: 0.9524 | Iters: 180
🏃 View run Run_Semilla_16 at: http://127.0.0.1:5000/#/experiments/1/runs/7440de703fe74e0d8eecfff7abbdfc5b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_17] Acc: 0.9737 | Recall: 0.9286 | Iters: 197
🏃 View run Run_Semilla_17 at: http://127.0.0.1:5000/#/experiments/1/runs/7e13522cbfe6438a9be3343c87ae3766
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_18] Acc: 0.9737 | Recall: 0.9286 | Iters: 189
🏃 View run Run_Semilla_18 at: http://127.0.0.1:5000/#/experiments/1/runs/bae7dec4a9fb46e6a2dc68d696f955ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[Run_Semilla_19] Acc: 0.9825 | Recall: 

(0.9736842105263158, 0.9285714285714286)

In [19]:
# FASE 5: Analizar, exportar y cerrar túnel
runs_df = mlflow.search_runs(experiment_names=["IEI097-L2.12-Naveas"])
runs_df.to_csv('runs.csv', index=False)
print("Historial exportado a 'runs.csv'. ¡Descárgalo ahora junto con las capturas!")

# IMPORTANTE: Tomar capturas de MLflow AHORA antes de ejecutar esta línea
ngrok.kill()
# *Descomenta ngrok.kill() solo cuando hayas terminado de tomar todas tus capturas*

Historial exportado a 'runs.csv'. ¡Descárgalo ahora junto con las capturas!


# 💬 Preguntas de Cierre

**1. ¿Qué cambió al escalar las variables?**
Cambió dramáticamente el rendimiento predictivo y la estabilidad. En mi *Run A* (sin escalar), el modelo arroja un rendimiento muy pobre y tarda muchas iteraciones, ya que variables masivas (como `area_mean`) saturan la red. En el *Run B* (escalado con StandardScaler), tanto la exactitud como la sensibilidad aumentan drásticamente. Al igualar la magnitud (media 0, desviación 1), el descenso del gradiente encuentra el mínimo global mucho más rápido y sin saturar las neuronas.

**2. ¿Más capas y neuronas mejoraron el resultado?**
Al comparar el *Run B* (1 capa de 30) con el *Run C* (2 capas: 64, 32), la mejora puede ser marginal o inexistente en el Test. Esto se debe a que un dataset de 569 filas no requiere una arquitectura tan profunda. Demasiadas neuronas pueden llevar a sobreajuste (overfitting). Además, si la diferencia entre *B* y *C* cae dentro del margen de variabilidad de mis semillas, no se puede afirmar estadísticamente que *C* sea estructuralmente mejor.

**3. Con tus 5 semillas, ¿qué tan grande fue la variación?**
La exactitud y sensibilidad variaron notablemente entre el mínimo y máximo de las 5 semillas. Esto demuestra que la inicialización de los pesos y el orden de los datos afectan el mínimo local al que llega el MLP. Si esta variación es mayor que la diferencia de métricas entre dos arquitecturas, implica que no podemos declarar a un modelo como "mejor" basándonos en una sola ejecución; necesitamos evaluar promedios de múltiples iteraciones (validación cruzada).

**4. ¿Qué error es más costoso en diagnóstico de cáncer?**
Un **falso negativo** es, por lejos, el error más costoso (predecir que un tumor es benigno cuando en realidad es maligno), ya que el paciente pierde un tratamiento médico vital y su vida corre riesgo. Por esto, la métrica a priorizar obligatoriamente es la **Sensibilidad (Recall)** de la clase maligna, asegurando que la mayor cantidad posible de tumores peligrosos sean detectados, incluso a costa de generar algunos falsos positivos.

**5. ¿Qué información de MLflow no tendrías con solo print()?**
Con solo `print()`, perdería la **observabilidad** total:
*   No tendría el historial visual de la curva de pérdida (`train_loss`) mapeada por épocas.
*   No tendría centralizado el registro de matriz de confusión (artefactos) de experimentos pasados.
*   No tendría registro automático de todo el entorno del experimento. Para replicarlo, tendría que registrar explícitamente las versiones de las librerías, la configuración de hiperparámetros y el hardware, algo que MLflow empaqueta de forma automática.